In [1]:
import pandas as pd

df = pd.read_csv('student_loan_complaints.csv')

df = df[~df['company_response_to_consumer'].isin(['In progress', 'Untimely response'])].copy()

issue_mapping = {
    'Repaying your loan': 'Dealing with my lender or servicer',
    'Problems when you are unable to pay': "Can't repay my loan"
}
df['issue'] = df['issue'].replace(issue_mapping)

def simplify_response(response):
    if response == 'Closed with explanation':
        return 'Explanation'
    elif response in ['Closed with monetary relief', 'Closed with non-monetary relief', 'Closed with relief']:
        return 'Relief'
    else:
        return 'Other'

df['company_response_to_consumer'] = df['company_response_to_consumer'].apply(simplify_response)

df['consumer_disputed?'] = df['consumer_disputed?'].map({'Yes': 1, 'No': 0})

df['tags'] = df['tags'].fillna('None')

df['date_received'] = pd.to_datetime(df['date_received'])

df = df.sort_values(by='date_received')

df.to_csv('student_loan_complaints_cleaned_sorted.csv', index=False)

In [2]:
import pandas as pd

companies = [
    'Navient Solutions, Inc.',
    'AES/PHEAA',
    'Wells Fargo & Company',
    'Sallie Mae',
    'JPMorgan Chase & Co.',
    'Discover',
    'Citibank',
    'Genesis Lending',
    'ACS Education Services',
    'KeyBank NA'
]

data = {
    'company': companies,
    
    'institution_type': [
        'Specialist', # Navient
        'Specialist', # AES/PHEAA
        'Bank',       # Wells Fargo
        'Specialist', # Sallie Mae (Post-split primarily a lender/specialist)
        'Bank',       # Chase
        'Bank',       # Discover
        'Bank',       # Citi
        'Specialist', # Genesis
        'Specialist', # ACS
        'Bank'        # KeyBank
    ],
    
    # 1 = Yes, 0 = No
    'enforcement_action': [
        1, # Navient: 2014 DOJ/FDIC settlement ($97M)
        0, # AES: 大罚单在 2012之前或2016之后
        0, # Wells Fargo: 学贷大罚单在 2016年8月 (outside window)
        1, # Sallie Mae: 2014 DOJ settlement (同 Navient)
        0, # Chase: 主要是信用卡和房贷罚单
        1, # Discover: 2015 CFPB order ($18.5M) for student loans
        0, # Citi: 学贷罚单在 2017
        0, # Genesis: 无重大公开记录
        0, # ACS: 主要是后续年份
        0  # KeyBank: 无重大记录
    ],

    'sl_assets_billions': [
        134.0, # Navient: 巨头，持有+管理极大份额
        100.0, # AES/PHEAA: 管理资产极大 (Servicing portfolio)
        11.9,  # Wells Fargo: 相比之下很小
        8.5,   # Sallie Mae: 拆分后专注于新贷款，规模变小
        12.0,  # Chase: 已经停止发放新学贷，在消化旧账
        8.6,   # Discover: 激进增长中，但绝对值仍小
        3.5,   # Citi: 正在退出该市场
        0.5,   # Genesis: Niche player (估算)
        15.0,  # ACS: Xerox子公司，管理部分联邦贷款
        2.5    # KeyBank: 区域性银行
    ]
}

df_external = pd.DataFrame(data)

df_external.to_csv('external_data_top10.csv', index=False)

In [5]:
import pandas as pd

df_main = pd.read_csv('student_loan_complaints_cleaned.csv')
df_external = pd.read_csv('external_data_top10.csv')

df_final = pd.merge(df_main, df_external, on='company', how='inner')

df_final.to_csv('final_data.csv', index=False)

In [9]:
import pandas as pd

df = pd.read_csv('final_data.csv')

if 'sub_product' in df.columns:
    df = df.drop(columns=['sub_product'])

def group_submission(val):
    if val == 'Web':
        return 'Web'
    elif val == 'Referral':
        return 'Referral' # VIP channel
    else:
        return 'Other' # Phone, Postal mail, Fax (High friction)

df['submission_group'] = df['submitted_via'].apply(group_submission)

df['date_received'] = pd.to_datetime(df['date_received'])
df['complaint_year'] = df['date_received'].dt.year

df.to_csv('final_data_cleaned.csv', index=False)

In [ ]:
final_vars = ['issue', 'submission_group', 'complaint_year']

print("最终变量分布检查:")
for var in final_vars:
    print(f"\n--- {var} ---")
    print(df[var].value_counts(normalize=True))


最终变量分布检查:

--- issue ---
issue
Dealing with my lender or servicer    0.617134
Can't repay my loan                   0.354668
Getting a loan                        0.028198
Name: proportion, dtype: float64

--- submission_group ---
submission_group
Web         0.869228
Referral    0.066308
Other       0.064464
Name: proportion, dtype: float64

--- complaint_year ---
complaint_year
2015    0.269919
2014    0.268920
2013    0.200231
2012    0.195083
2016    0.065847
Name: proportion, dtype: float64


In [3]:
import pandas as pd
df = pd.read_csv('student_loan_complaints_cleaned.csv')
df["company"].value_counts().head(30)

company
Navient Solutions, Inc.                                  6625
AES/PHEAA                                                1568
Wells Fargo & Company                                     994
Sallie Mae                                                786
JPMorgan Chase & Co.                                      711
Discover                                                  606
Citibank                                                  505
Genesis Lending                                           445
ACS Education Services                                    416
KeyBank NA                                                359
Transworld Systems Inc.                                   239
First Associates Loan Servicing LLC                       191
Expert Global Solutions, Inc.                             149
Nelnet                                                    129
Higher Education Student Assistance Authority (HESAA)     125
Citizens Financial Group, Inc.                             80


In [15]:
import pandas as pd

# 1. 读取你的 CSV 数据集 (请替换为你的实际文件名)
df = pd.read_csv('student_loan_complaints_cleaned.csv')

# 2. 定义你想要查找的关键关键词
# 建议只写核心词，忽略 Inc. 或 LLC 这种后缀
target_keywords = [
    'cornerstone', 
    'Granite', 
    'oklahoma', 
    'edfinancial',
    'vermont',
    'iowa',
    'texas'
]

# 3. 在 'Company' 列中查找这些关键词 (忽略大小写)
for keyword in target_keywords:
    print(f"--- 正在搜索关键词: {keyword} ---")
    # 使用 str.contains 模糊查找，case=False 忽略大小写，na=False 处理缺失值
    matches = df[df['company'].str.contains(keyword, case=False, na=False)]['company'].unique()
    
    if len(matches) > 0:
        for name in matches:
            print(f"找到匹配名称: {name}")
    else:
        print("没有找到匹配结果。")
    print("\n")

--- 正在搜索关键词: cornerstone ---
没有找到匹配结果。


--- 正在搜索关键词: Granite ---
找到匹配名称: Granite State Management & Resources


--- 正在搜索关键词: oklahoma ---
没有找到匹配结果。


--- 正在搜索关键词: edfinancial ---
找到匹配名称: Edfinancial Services


--- 正在搜索关键词: vermont ---
找到匹配名称: Vermont Student Assistance Corporation


--- 正在搜索关键词: iowa ---
找到匹配名称: Iowa Student Loan


--- 正在搜索关键词: texas ---
找到匹配名称: Texas Higher Education Coordinating Board
找到匹配名称: Texas Guaranteed




In [17]:
import pandas as pd

# 1. 加载你的数据集 (请确保文件名与你电脑上的文件名一致)
file_path = 'student_loan_complaints_cleaned.csv' 
df = pd.read_csv(file_path)

# 2. 定义你的 20 家目标公司名单 (根据你之前的确定好的名单)
target_companies = [
    'Navient Solutions, Inc.', 'AES/PHEAA', 'Great Lakes', 'Nelnet', 'MOHELA',
    'Granite State Management & Resources', 'Edfinancial Services', 
    'Vermont Student Assistance Corporation', 'Iowa Student Loan', 'Texas Guaranteed',
    'Wells Fargo & Company', 'Discover', 'Sallie Mae', 'JPMorgan Chase & Co.', 
    'Citibank', 'PNC Bank N.A.', 'Citizens Financial Group, Inc.', 
    'SunTrust Banks, Inc.', 'KeyBank NA', 'U.S. Bancorp'
]

# 3. 计算全集的总投诉量
total_complaints_all = len(df)

# 4. 筛选出这 20 家公司的数据
df_20 = df[df['company'].isin(target_companies)].copy()
total_complaints_20 = len(df_20)

# 5. 统计每家公司的具体数量并排序
company_counts = df_20['company'].value_counts().reset_index()
company_counts.columns = ['company', 'Complaint_Count']

# 6. 计算占比
company_counts['Percentage_of_Total'] = (company_counts['Complaint_Count'] / total_complaints_all) * 100
overall_coverage = (total_complaints_20 / total_complaints_all) * 100

# 7. 打印结果
print(f"整个数据集总投诉量: {total_complaints_all}")
print(f"这 20 家公司的总投诉量: {total_complaints_20}")
print(f"这 20 家公司覆盖了全美约 {overall_coverage:.2f}% 的投诉量\n")
print("各公司详细统计:")
print(company_counts)

整个数据集总投诉量: 15724
这 20 家公司的总投诉量: 12732
这 20 家公司覆盖了全美约 80.97% 的投诉量

各公司详细统计:
                                   company  Complaint_Count  \
0                  Navient Solutions, Inc.             6625   
1                                AES/PHEAA             1568   
2                    Wells Fargo & Company              994   
3                               Sallie Mae              786   
4                     JPMorgan Chase & Co.              711   
5                                 Discover              606   
6                                 Citibank              505   
7                               KeyBank NA              359   
8                                   Nelnet              129   
9           Citizens Financial Group, Inc.               80   
10                                  MOHELA               66   
11                            U.S. Bancorp               66   
12                             Great Lakes               65   
13                           PNC Bank N.A. 